In [1]:
%pip install -U ustreasurycurve pandas polars pyarrow

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\Vladyslav\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import ustreasurycurve as ustcurve
# 1. Fetch the full nominal yield curve history (from 1990 to present)
df_nominal = ustcurve.nominal_rates()

# 2. Or pull a specific historical window
df_window = ustcurve.nominal_rates('2026-09-01', '2026-09-05')

print(df_window.tail())

        date    1m  1.5m    2m    3m    4m    6m    1y    2y    3y    5y  \
0 2026-09-01  3.85  3.88  3.89  3.92  3.97  4.00  4.18  4.39  4.46  4.55   
1 2026-09-02  3.83  3.87  3.89  3.92  4.02  4.00  4.16  4.39  4.45  4.54   
2 2026-09-03  3.83  3.82  3.91  3.89  3.99  3.95  4.11  4.34  4.41  4.52   
3 2026-09-04  3.79  3.83  3.90  3.91  4.00  3.98  4.13  4.37  4.45  4.54   

     7y   10y   20y   30y  
0  4.66  4.79  5.27  5.27  
1  4.66  4.79  5.27  5.27  
2  4.63  4.77  5.25  5.25  
3  4.65  4.78  5.25  5.24  


In [3]:
df = ustcurve.nominal_rates(
    '2026-09-01',
    '2026-09-22'
)

df = df[['date', '2y', '10y']].copy()
df

,date,2y,10y
0,2026-09-01,4.39,4.79
1,2026-09-02,4.39,4.79
2,2026-09-03,4.34,4.77
3,2026-09-04,4.37,4.78
4,2026-09-08,4.39,4.80
5,2026-09-09,4.43,4.83
6,2026-09-10,4.56,4.95
7,2026-09-11,4.63,4.96
8,2026-09-14,4.65,4.97
9,2026-09-15,4.67,5.00


In [4]:
import pandas as pd
import ustreasurycurve as ustcurve

df = ustcurve.nominal_rates(
    '2026-08-31',
    '2026-09-22'
)

date2y10y = df[['date', '2y', '10y']].copy()

date2y10y['date'] = pd.to_datetime(date2y10y['date'])
date2y10y = date2y10y.sort_values('date')

# Shift yields by one available observation
date2y10y['2y'] = date2y10y['2y'].shift(1)
date2y10y['10y'] = date2y10y['10y'].shift(1)

# Remove Aug 31 because it has no previous observation in our query
date2y10y = date2y10y[
    date2y10y['date'] >= '2026-09-01'
].reset_index(drop=True)

print(date2y10y)

         date    2y   10y
0  2026-09-01  4.34  4.75
1  2026-09-02  4.39  4.79
2  2026-09-03  4.39  4.79
3  2026-09-04  4.34  4.77
4  2026-09-08  4.37  4.78
5  2026-09-09  4.39  4.80
6  2026-09-10  4.43  4.83
7  2026-09-11  4.56  4.95
8  2026-09-14  4.63  4.96
9  2026-09-15  4.65  4.97
10 2026-09-16  4.67  5.00
11 2026-09-17  4.74  5.01
12 2026-09-18  4.67  4.94
13 2026-09-21  4.76  5.01
14 2026-09-22  4.76  4.96


In [5]:
import pandas as pd
import ustreasurycurve as ustcurve

# Get enough history to have an observation before Sep 1
df = ustcurve.nominal_rates(
    '2026-08-01',
    '2026-09-22'
)

# Select the columns
date2y10y = df[['date', '2y', '10y']].copy()

# Ensure datetime
date2y10y['date'] = pd.to_datetime(date2y10y['date'])

# Sort
date2y10y = date2y10y.sort_values('date')

# Create a complete calendar
calendar = pd.date_range(
    start='2026-09-01',
    end='2026-09-22',
    freq='D'
)

# Put Treasury observations onto the complete calendar
date2y10y = (
    date2y10y
    .set_index('date')
    .reindex(calendar)
)

# Remember the actual Treasury observation date
date2y10y['source_date'] = date2y10y.index.to_series().where(
    date2y10y['2y'].notna()
)

# Carry the last available Treasury observation forward
date2y10y['source_date'] = date2y10y['source_date'].ffill()
date2y10y[['2y', '10y']] = date2y10y[['2y', '10y']].ffill()

# Put date back as a column
date2y10y = date2y10y.reset_index()
date2y10y = date2y10y.rename(columns={'index': 'date'})

print(date2y10y)

         date    2y   10y source_date
0  2026-09-01  4.39  4.79  2026-09-01
1  2026-09-02  4.39  4.79  2026-09-02
2  2026-09-03  4.34  4.77  2026-09-03
3  2026-09-04  4.37  4.78  2026-09-04
4  2026-09-05  4.37  4.78  2026-09-04
5  2026-09-06  4.37  4.78  2026-09-04
6  2026-09-07  4.37  4.78  2026-09-04
7  2026-09-08  4.39  4.80  2026-09-08
8  2026-09-09  4.43  4.83  2026-09-09
9  2026-09-10  4.56  4.95  2026-09-10
10 2026-09-11  4.63  4.96  2026-09-11
11 2026-09-12  4.63  4.96  2026-09-11
12 2026-09-13  4.63  4.96  2026-09-11
13 2026-09-14  4.65  4.97  2026-09-14
14 2026-09-15  4.67  5.00  2026-09-15
15 2026-09-16  4.74  5.01  2026-09-16
16 2026-09-17  4.67  4.94  2026-09-17
17 2026-09-18  4.76  5.01  2026-09-18
18 2026-09-19  4.76  5.01  2026-09-18
19 2026-09-20  4.76  5.01  2026-09-18
20 2026-09-21  4.76  4.96  2026-09-21
21 2026-09-22  4.71  4.96  2026-09-22


In [6]:
import pandas as pd
import ustreasurycurve as ustcurve

# Get enough history to have a previous observation for Sep 1
df = ustcurve.nominal_rates(
    '2026-08-01',
    '2026-09-22'
)

# Select 2Y and 10Y
date2y10y = df[['date', '2y', '10y']].copy()

# Ensure proper date type and sorting
date2y10y['date'] = pd.to_datetime(date2y10y['date'])
date2y10y = date2y10y.sort_values('date')

# -------------------------------------------------------
# 1. Create weekdays only
# -------------------------------------------------------

weekdays = pd.date_range(
    start='2026-09-01',
    end='2026-09-22',
    freq='B'
)

# -------------------------------------------------------
# 2. Reindex to weekdays
# -------------------------------------------------------

date2y10y = (
    date2y10y
    .set_index('date')
    .reindex(weekdays)
)

# -------------------------------------------------------
# 3. Keep track of the actual Treasury observation date
# -------------------------------------------------------

date2y10y['source_date'] = date2y10y.index.to_series().where(
    date2y10y['2y'].notna()
)

# -------------------------------------------------------
# 4. For missing weekdays (holidays), use previous
#    available Treasury observation
# -------------------------------------------------------

date2y10y['source_date'] = date2y10y['source_date'].ffill()

date2y10y[['2y', '10y']] = (
    date2y10y[['2y', '10y']]
    .ffill()
)

# -------------------------------------------------------
# 5. Convert index back to column
# -------------------------------------------------------

date2y10y = date2y10y.reset_index()
date2y10y = date2y10y.rename(columns={'index': 'date'})

print(date2y10y)

         date    2y   10y source_date
0  2026-09-01  4.39  4.79  2026-09-01
1  2026-09-02  4.39  4.79  2026-09-02
2  2026-09-03  4.34  4.77  2026-09-03
3  2026-09-04  4.37  4.78  2026-09-04
4  2026-09-07  4.37  4.78  2026-09-04
5  2026-09-08  4.39  4.80  2026-09-08
6  2026-09-09  4.43  4.83  2026-09-09
7  2026-09-10  4.56  4.95  2026-09-10
8  2026-09-11  4.63  4.96  2026-09-11
9  2026-09-14  4.65  4.97  2026-09-14
10 2026-09-15  4.67  5.00  2026-09-15
11 2026-09-16  4.74  5.01  2026-09-16
12 2026-09-17  4.67  4.94  2026-09-17
13 2026-09-18  4.76  5.01  2026-09-18
14 2026-09-21  4.76  4.96  2026-09-21
15 2026-09-22  4.71  4.96  2026-09-22


In [7]:
%pip install -U pandas requests

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\Vladyslav\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [8]:
import pandas as pd
import requests
from io import StringIO


def get_ecb_aaa_yield_curve(
    start='2026-09-01',
    end='2026-09-22',
    carry_forward_holidays=True
):
    """
    Download ECB euro-area AAA yield curve data and return
    2Y and 10Y spot rates.

    Output columns:
        date
        2Y
        10Y
        source_date

    ECB holidays:
        Weekdays with no ECB observation are retained and,
        if carry_forward_holidays=True, filled with the
        previous available ECB observation.

    Weekends:
        Excluded.
    """

    # ECB series for 2Y and 10Y spot rates
    series = "SR_2Y+SR_10Y"

    url = (
        "https://data-api.ecb.europa.eu/service/data/"
        "YC/B.U2.EUR.4F.G_N_A.SV_C_YM."
        f"{series}"
    )

    params = {
        "startPeriod": start,
        "endPeriod": end,
        "format": "csvdata"
    }

    response = requests.get(url, params=params, timeout=60)
    response.raise_for_status()

    # Read ECB CSV
    df = pd.read_csv(StringIO(response.text))

    # Clean column names
    df.columns = df.columns.str.strip()

    # Dates and values
    df["TIME_PERIOD"] = pd.to_datetime(df["TIME_PERIOD"])
    df["OBS_VALUE"] = pd.to_numeric(df["OBS_VALUE"], errors="coerce")

    # Keep only 2Y and 10Y
    df = df[df["DATA_TYPE_FM"].isin(["SR_2Y", "SR_10Y"])].copy()

    # Convert to wide format
    df = (
        df.pivot(
            index="TIME_PERIOD",
            columns="DATA_TYPE_FM",
            values="OBS_VALUE"
        )
        .rename(columns={
            "SR_2Y": "2Y",
            "SR_10Y": "10Y"
        })
    )

    # Make sure both columns exist
    for col in ["2Y", "10Y"]:
        if col not in df.columns:
            df[col] = pd.NA

    # Sort dates
    df = df.sort_index()

    # ---------------------------------------------------------
    # Create a weekday-only calendar.
    #
    # This excludes Saturday/Sunday but keeps weekdays on which
    # the ECB did not publish an observation (e.g. TARGET holidays).
    # ---------------------------------------------------------

    calendar = pd.date_range(
        start=start,
        end=end,
        freq="B"
    )

    df = df.reindex(calendar)
    df.index.name = "date"

    # ---------------------------------------------------------
    # Remember the actual ECB observation date
    # ---------------------------------------------------------

    df["source_date"] = pd.Series(
        df.index.where(df["2Y"].notna()),
        index=df.index
    ).ffill()

    # ---------------------------------------------------------
    # Carry previous ECB observation onto missing weekdays
    # ---------------------------------------------------------

    if carry_forward_holidays:
        df[["2Y", "10Y"]] = df[["2Y", "10Y"]].ffill()

    # Reset index
    df = df.reset_index()

    return df[["date", "2Y", "10Y", "source_date"]]

In [11]:
import pandas as pd
import requests
from io import StringIO
import time


def _get_ecb_series(series, start, end, retries=3):
    """
    Download one ECB yield-curve series with retries.
    """

    url = (
        "https://data-api.ecb.europa.eu/service/data/"
        f"YC/B.U2.EUR.4F.G_N_A.SV_C_YM.{series}"
    )

    params = {
        "startPeriod": start,
        "endPeriod": end,
        "format": "csvdata"
    }

    for attempt in range(retries):
        try:
            response = requests.get(
                url,
                params=params,
                timeout=90
            )

            response.raise_for_status()

            return pd.read_csv(StringIO(response.text))

        except requests.RequestException as e:

            if attempt == retries - 1:
                raise RuntimeError(
                    f"ECB API failed after {retries} attempts.\n"
                    f"Series: {series}\n"
                    f"Error: {e}"
                )

            # Wait before retrying
            time.sleep(3 * (attempt + 1))


def get_ecb_aaa_yield_curve(
    start='2026-09-01',
    end='2026-09-22',
    carry_forward_holidays=True
):

    # ---------------------------------------------------------
    # 1. Download the two AAA spot-rate series separately
    # ---------------------------------------------------------

    df_2y = _get_ecb_series(
        "SR_2Y",
        start,
        end
    )

    df_10y = _get_ecb_series(
        "SR_10Y",
        start,
        end
    )

    # ---------------------------------------------------------
    # 2. Keep only the fields we need
    # ---------------------------------------------------------

    df_2y = df_2y[
        ["TIME_PERIOD", "OBS_VALUE"]
    ].copy()

    df_10y = df_10y[
        ["TIME_PERIOD", "OBS_VALUE"]
    ].copy()

    # Dates
    df_2y["TIME_PERIOD"] = pd.to_datetime(df_2y["TIME_PERIOD"])
    df_10y["TIME_PERIOD"] = pd.to_datetime(df_10y["TIME_PERIOD"])

    # Values
    df_2y["OBS_VALUE"] = pd.to_numeric(
        df_2y["OBS_VALUE"],
        errors="coerce"
    )

    df_10y["OBS_VALUE"] = pd.to_numeric(
        df_10y["OBS_VALUE"],
        errors="coerce"
    )

    # Rename
    df_2y = df_2y.rename(
        columns={
            "TIME_PERIOD": "date",
            "OBS_VALUE": "2Y"
        }
    )

    df_10y = df_10y.rename(
        columns={
            "TIME_PERIOD": "date",
            "OBS_VALUE": "10Y"
        }
    )

    # ---------------------------------------------------------
    # 3. Combine 2Y and 10Y
    # ---------------------------------------------------------

    df = pd.merge(
        df_2y,
        df_10y,
        on="date",
        how="outer"
    )

    df = df.sort_values("date")

    # ---------------------------------------------------------
    # 4. Create weekday-only calendar
    #
    # Mon-Fri are included.
    # Saturday/Sunday are excluded.
    #
    # ECB/TARGET holidays appear as weekdays with no
    # observation, which lets us carry forward the previous
    # ECB observation.
    # ---------------------------------------------------------

    weekdays = pd.date_range(
        start=start,
        end=end,
        freq="B"
    )

    df = (
        df.set_index("date")
          .reindex(weekdays)
    )

    df.index.name = "date"

    # ---------------------------------------------------------
    # 5. Record the actual ECB observation date
    # ---------------------------------------------------------

    df["source_date"] = pd.Series(
        df.index.where(df["2Y"].notna()),
        index=df.index
    ).ffill()

    # ---------------------------------------------------------
    # 6. Carry previous observation onto non-publication days
    # ---------------------------------------------------------

    if carry_forward_holidays:

        df[["2Y", "10Y"]] = (
            df[["2Y", "10Y"]]
            .ffill()
        )

    # ---------------------------------------------------------
    # 7. Return clean DataFrame
    # ---------------------------------------------------------

    return df.reset_index()[
        ["date", "2Y", "10Y", "source_date"]
    ]

In [12]:
df_euro = get_ecb_aaa_yield_curve(
    start='2026-09-01',
    end='2026-09-22'
)

df_euro

,date,2Y,10Y,source_date
0,2026-09-01,2.876001,3.355152,2026-09-01
1,2026-09-02,2.913206,3.391851,2026-09-02
2,2026-09-03,2.885664,3.364624,2026-09-03
3,2026-09-04,2.876637,3.346497,2026-09-04
4,2026-09-07,2.913307,3.389493,2026-09-07
5,2026-09-08,2.919754,3.378456,2026-09-08
6,2026-09-09,2.976293,3.426566,2026-09-09
7,2026-09-10,3.100596,3.503209,2026-09-10
8,2026-09-11,3.127569,3.522729,2026-09-11
9,2026-09-14,3.227780,3.540940,2026-09-14


In [10]:
df_euro = get_ecb_aaa_yield_curve(
    start='2026-09-01',
    end='2026-09-22'
)

df_euro

HTTPError: 504 Server Error: Gateway Time-out for url: https://data-api.ecb.europa.eu/service/data/YC/B.U2.EUR.4F.G_N_A.SV_C_YM.SR_2Y+SR_10Y?startPeriod=2026-09-01&endPeriod=2026-09-22&format=csvdata